# Finite-size corrections with Yeh-Hummer

When performing molecular dynamics simulations with periodic boundary conditions, the calculated diffusion coefficients are affected by finite-size effects. The `kinisi` package includes tools to apply the Yeh-Hummer finite-size correction, which extrapolates the diffusion coefficient to the infinite system size limit.

In this tutorial, we will demonstrate how to use the `YehHummer` class to correct diffusion coefficients obtained from simulations with different box sizes.

<div class="alert alert-info">

**Note**

The Yeh-Hummer correction is based on the work by Yeh & Hummer (J. Phys. Chem. B 2004, 108, 15873-15879). The method assumes that the finite-size effects follow a linear relationship with the inverse box length.
</div>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipp as sc

from kinisi.yeh_hummer import YehHummer

np.random.seed(42)

## Setting up the data

For this tutorial, we will use the TIP3P water diffusion data from the original Yeh & Hummer paper. This data represents diffusion coefficients calculated from molecular dynamics simulations of water at different box sizes.

In [ ]:
# TIP3P water data from Yeh & Hummer paper
box_lengths = np.array([18.58, 23.42, 29.51, 37.19, 46.86])  # Angstroms
D_values = np.array([4.884e-5, 5.123e-5, 5.315e-5, 5.466e-5, 5.590e-5])  # cm^2/s
D_errors = np.array([0.032e-5, 0.027e-5, 0.014e-5, 0.011e-5, 0.013e-5])  # cm^2/s

We need to organize this data into a `scipp.DataArray` with the appropriate coordinates and variances.

In [ ]:
td = sc.DataArray(
    data=sc.array(dims=['system'], values=D_values, variances=D_errors**2, unit='cm^2/s'),
    coords={'box_length': sc.Variable(dims=['system'], values=box_lengths, unit='angstrom')},
)
td

## Creating the YehHummer object

With our data prepared, we can now create a `YehHummer` object. The temperature is required for the viscosity calculation. You can optionally provide bounds for the fitting parameters.

In [ ]:
# Create YehHummer object with temperature 298 K
yh = YehHummer(td, temperature=298)

## MCMC sampling for uncertainty estimation

To obtain full probability distributions for the parameters, we can perform Markov chain Monte Carlo (MCMC) sampling:

In [ ]:
yh.mcmc(n_samples=500, n_walkers=16)

After MCMC sampling, we can visualize the probability distributions for the fitted parameters:

In [ ]:
from corner import corner

corner(
    np.array([i.values for i in yh.flatchain.values()]).T,
    labels=[' / '.join([k, str(v.unit)]) for k, v in yh.flatchain.items()],
)
plt.show()

## Accessing the results

The infinite-system diffusion coefficient and shear viscosity are now available as probability distributions:

In [ ]:
yh.D_infinite

In [ ]:
yh.shear_viscosity

## Plotting the results

The `YehHummer` class includes a built-in plotting method that shows the data points, the fitted relationship, and credible intervals. The plot extrapolates to infinite box size (1/L → 0):

In [ ]:
yh.plot()
plt.show()

## Understanding the results

The Yeh-Hummer correction provides two important quantities:

1. **Infinite-system diffusion coefficient (`D_∞`)**: This is the extrapolated diffusion coefficient for an infinite system.

2. **Shear viscosity (`η`)**: This is estimated from the slope of the linear relationship:

$$D_{\text{PBC}} = D_\infty - \frac{k_B T \xi}{6\pi\eta L}$$

where:
- $D_{\text{PBC}}$ is the diffusion coefficient from the periodic simulation
- $D_\infty$ is the infinite-system diffusion coefficient
- $k_B$ is the Boltzmann constant
- $T$ is the temperature
- $\xi$ constant from Ewald summation (2.837297 for cubic boxes)
- $\eta$ is the shear viscosity
- $L$ is the box length